In [1]:
# 1. CLEAN INSTALLATION (Run this first)
!pip install --no-cache-dir speechbrain==1.0.1 moshi torchaudio torch

# 2. THE MONKEY PATCH (Run this immediately after installation)
import torchaudio
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["sox_io"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 22.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 315.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 162.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 376.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 172.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 225.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 348.4 MB/s eta 0:00:00
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully

In [2]:
import os
import glob
import torch
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier
# Assuming standard Moshi library imports
from moshi.models import Mimi
from tqdm.notebook import tqdm

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
ESD_DATA_DIR = "/kaggle/input/datasets/nguyenthanhlim/emotional-speech-dataset-esd/Emotion Speech Dataset" 
OUTPUT_DIR = "./Processed_Tensors"

# ECAPA-TDNN requires 16kHz, Mimi typically expects 24kHz
SR_ECAPA = 16000
SR_MIMI = 24000

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading ECAPA-TDNN Biometric Extractor...")
identity_extractor = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb", 
    savedir="pretrained_models/ecapa"
).to(device)
identity_extractor.eval()

print("Loading Kyutai Mimi Codec...")
# Load the Mimi codec to pre-extract the 8 discrete RVQ codebooks
mimi_codec = Mimi.from_pretrained().to(device)
mimi_codec.eval()

# ==========================================
# 2. THE 11-20 EXTRACTION PIPELINE
# ==========================================
def process_esd_dataset():
    wav_files = glob.glob(os.path.join(ESD_DATA_DIR, "**/*.wav"), recursive=True)
    
    # 🛠️ Filter explicitly for English Speakers (11 through 20)
    valid_files = []
    for filepath in wav_files:
        speaker_id_str = filepath.split(os.sep)[-3] # e.g., '0011'
        if speaker_id_str.isdigit():
            if 11 <= int(speaker_id_str) <= 20:
                valid_files.append(filepath)
                
    print(f"Filtered to {len(valid_files)} English audio files (Speakers 11-20).")
    
    for filepath in tqdm(valid_files):
        filename = os.path.basename(filepath).replace(".wav", ".pt")
        
        # Load raw audio
        waveform, sr = torchaudio.load(filepath)
        waveform = waveform.to(device)
        
        with torch.no_grad():
            # --- Biometric Extraction (16kHz) ---
            if sr != SR_ECAPA:
                wav_ecapa = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=SR_ECAPA)
            else:
                wav_ecapa = waveform
                
            id_vector = identity_extractor.encode_batch(wav_ecapa)
            id_vector = id_vector.squeeze() # Shape: [192]
            
            # --- Mimi Token Extraction (24kHz) ---
            if sr != SR_MIMI:
                wav_mimi = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=SR_MIMI)
            else:
                wav_mimi = waveform
                
            # Encode audio into the 8 RVQ codebooks (Shape: [8, seq_len])
            mimi_tokens = mimi_codec.encode(wav_mimi.unsqueeze(0)).squeeze(0)
            
        # ==========================================
        # 3. HYBRID TENSOR PACKAGING
        # ==========================================
        # Cast identity vector to bfloat16; tokens remain standard integers
        hybrid_tensor = {
            "identity_vector": id_vector.cpu().to(torch.bfloat16),
            "mimi_tokens": mimi_tokens.cpu() # Shape: [8, seq_len]
        }
        
        # Save straight to the output directory
        torch.save(hybrid_tensor, os.path.join(OUTPUT_DIR, filename))

if __name__ == "__main__":
    process_esd_dataset()
    print("Data Factory Complete. Tensors are ready for the V2.2 Training Engine.")

Initializing Notebook 1: The MagicHub Data Factory...
Loading Mimi Weights & Speaker Classifier...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

hyperparams.yaml: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/speechbrain/utils/autocast.py:68: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

...

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Found 16 files. Starting extraction...


Processing MagicHub: 100%|██████████| 16/16 [00:36<00:00,  2.30s/it]

SUCCESS: Processed 16 files.


In [3]:
!zip -r file.zip /kaggle/working



  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/processed_tensors/ (stored 0%)
  adding: kaggle/working/processed_tensors/Group0030_S004_0_ID024.pt (deflated 77%)
  adding: kaggle/working/processed_tensors/Group0078_S015_0_ID098.pt (deflated 85%)
  adding: kaggle/working/processed_tensors/Group0078_S005_0_ID099.pt (deflated 85%)
  adding: kaggle/working/processed_tensors/Group0078_S009_0_ID099.pt (deflated 75%)
  adding: kaggle/working/processed_tensors/Group0046_S009_0_ID023.pt (deflated 80%)
  adding: kaggle/working/processed_tensors/Group0046_S009_0_ID026.pt (deflated 72%)
  adding: kaggle/working/processed_tensors/Group0006_S001_0_ID164.pt (deflated 77%)
  adding: kaggle/working/processed_tensors/Group0078_S004_0_ID099.pt (deflated 78%)
  adding: kaggle/working/processed_tensors/Group0030_S004_0_ID029.pt (deflated 79%)
  adding: kaggle/working/processed_tensors/Group0006_S001_0_ID165.pt (deflated 81%)
  adding: kaggle/working/processed_tensors/Group0078_S015_0_ID099.